# Reward Comparison Analysis

This notebook rebuilds the selected analysis scope directly from `results/`, drops incomplete hyperparameter groups, exports merged CSVs, and generates reward comparison plots where:

- x-axis = episode
- y-axis = mean raw reward
- each line represents one selected comparison value averaged over the remaining retained dimensions
- log-based seed plots for `eval_success_rate` are generated as well


## Scope

- CartPole: `dqn_entropy` + `dqn_rnd`
- MountainCar: `dqn_entropy_mc_8000` + `dqn_rnd_mc_8000`
- Dense and sparse are analyzed separately in both tables and plots
- Incomplete groups are excluded before merge and plotting


In [ ]:
from pathlib import Path
import sys
import pandas as pd

if (Path.cwd() / "notebooks" / "results_analysis_helpers.py").exists():
    REPO_ROOT = Path.cwd()
    NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
elif (Path.cwd() / "results_analysis_helpers.py").exists():
    NOTEBOOKS_DIR = Path.cwd()
    REPO_ROOT = NOTEBOOKS_DIR.parent
else:
    raise RuntimeError("Run this notebook from the repo root or notebooks directory.")

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from results_analysis_helpers import EXPECTED_COUNTS, build_summary_tables, run_full_analysis

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)
EXPECTED_COUNTS


In [ ]:
artifacts = run_full_analysis(
    results_root=REPO_ROOT / "results",
    metrics_dir=REPO_ROOT / "metrics" / "analysis",
    visualizations_root=REPO_ROOT / "visualizations" / "analysis",
    logs_root=REPO_ROOT / "logs",
)
artifacts.counts


## Coverage Audit

These rows show which hyperparameter groups were kept or excluded before the merge.

In [ ]:
artifacts.coverage_df


## Summary Tables

Compact views of algorithm-level results and top retained configs, split by reward type.

In [ ]:
summary_tables = build_summary_tables(artifacts.summary_df)
summary_tables["overall_summary"]


In [ ]:
summary_tables["top_configs"].groupby(["analysis_cohort", "reward_type", "algorithm"], dropna=False).head(5)


## Generated Files

The merged CSVs are written under `metrics/analysis/` and the comparison plots are written under `visualizations/analysis/`.

In [ ]:
sorted(str(path.relative_to(REPO_ROOT)) for path in artifacts.plot_paths)


In [ ]:
from IPython.display import Image, display

sample_plots = [
    REPO_ROOT / "visualizations" / "analysis" / "cartpole" / "cartpole_dense_compare_by_seed.png",
    REPO_ROOT / "visualizations" / "analysis" / "cartpole" / "cartpole_dense_eval_success_compare_by_seed.png",
    REPO_ROOT / "visualizations" / "analysis" / "cartpole" / "cartpole_sparse_compare_by_seed.png",
    REPO_ROOT / "visualizations" / "analysis" / "mountaincar_mc8000" / "mountaincar_mc8000_dense_compare_by_seed.png",
    REPO_ROOT / "visualizations" / "analysis" / "mountaincar_mc8000" / "mountaincar_mc8000_dense_eval_success_compare_by_seed.png",
    REPO_ROOT / "visualizations" / "analysis" / "mountaincar_mc8000" / "mountaincar_mc8000_sparse_compare_by_seed.png",
]

for plot_path in sample_plots:
    display(Image(filename=str(plot_path)))
